In [ ]:
import importlib

from matplotlib import pyplot as plt
from os.path import join
import os
import seaborn as sns

from tqdm.notebook import tqdm

#### Custum libraries
import lib.algos_maxRSA as max_rsa
import lib.utils_RSA as rsa
import lib.utils_CKA as cka
from lib.algos import *


importlib.reload(rsa)
importlib.reload(cka)
importlib.reload(max_rsa)

In [ ]:
dataset = 'genLOC2'
arch = 'Ohran'
models = ['ego', 'saycam', 'imagenet', 'supervised', 'resnet', 'random']
#models  = ['ego', 'saycam']
path2activations = f'/home/alban/Documents/activations_datadriven/%s_{dataset}/'

rootsavedir = f'figures/{arch}_{dataset}/'
if not os.path.exists(rootsavedir):
    os.makedirs(rootsavedir)
if not os.path.exists(join(rootsavedir, 'subRDMs')):
    os.makedirs(join(rootsavedir, 'subRDMs'))
if not os.path.exists(join(rootsavedir, 'tSNE')):
    os.makedirs(join(rootsavedir, 'tSNE'))
if not os.path.exists(join(rootsavedir, 'subsets')):
    os.makedirs(join(rootsavedir, 'subsets'))

imagelists = {}
activations = {}
for model in models:
    with open(join(path2activations%model, 'imagepaths.txt'), 'r') as f:
        imagelists[model] = [line.strip() for line in f.readlines()]
    activations[model] = np.load(join(path2activations % model, 'cls_tokens.npy'))


imagelist = imagelists[model]
activations[model].shape

In [ ]:
#### Normalize vectors
for model in models:
    activations[model] = activations[model].reshape(activations[model].shape[0], activations[model].shape[1])
    norms = np.linalg.norm(activations[model], axis=1, keepdims=True)
    activations[model] = activations[model]/norms # normalization

In [ ]:
### check if images were shown in the same order
assert imagelists[models[0]] == imagelists[models[1]]
imagelist = imagelists[models[0]] # since they are the same, only consider one list

#### check if each category has the same number of images and list all categories in listcats
count = 0
cat = ''
listcat = list()
for i, imgp in enumerate(imagelist):
    current_cat = imgp.split('/')[-2]
    if i == 0:
        cat = current_cat
        listcat.append(current_cat)
    if cat != current_cat:
        cat = current_cat
        listcat.append(current_cat)
        count = 1
    else:
        count += 1

nb_per_cat = count # in val, 50 images per cate

nb_per_cat

In [ ]:
### reshape activations according to include categories
cat_activations = activations.copy()

for model in models:
    shape = activations[model].shape
    cat_activations[model] = activations[model].reshape(-1, nb_per_cat, shape[-1])

In [ ]:

savedir = f'results/compactness/fisher_discriminant2_ohran_{dataset}/'
if not os.path.exists(savedir):
    os.makedirs(savedir)
    sorted_compactness, sorted_compact_categories, compactness = max_rsa.compute_compactness(cat_activations, models, listcat, measure = 'Fisher_discriminant')
    np.save(join(savedir, 'compactness.npy'), compactness)
    np.save(join(savedir, 'sorted_compactness.npy'), sorted_compactness)
    np.save(join(savedir, 'sorted_compact_categories.npy'), sorted_compact_categories)
else: # if we computed and saved compactness before, let's just load it
    compactness = np.load(join(savedir, 'compactness.npy'), allow_pickle=True).item()
    sorted_compactness = np.load(join(savedir, 'sorted_compactness.npy'), allow_pickle=True).item()
    sorted_compact_categories = np.load(join(savedir, 'sorted_compact_categories.npy'), allow_pickle=True).item()

In [ ]:
fig_compactness, ax_compactness = max_rsa.plot_stats_one(sorted_compactness,models,  ['Categories', 'Compactness'])

In [ ]:
#### Compute correlations between the model's compactness


model_corr_matrix= np.zeros((len(models), len(models)))

for m1, model1 in enumerate(models):
    for m2, model2 in enumerate(models):
        model_corr_matrix[m1,m2] = np.round(np.corrcoef(compactness[model1], compactness[model2])[0,1], 2)
fig = plt.figure(figsize=(6, 6))
plt.rcParams['axes.grid'] = False
# Replace the plt.imshow() section with:
sns.heatmap(1 - model_corr_matrix,
        annot=True, fmt='.2f', cmap='grey', vmin=0, vmax = 1,
        xticklabels=models, yticklabels=models, cbar=False)
plt.xticks(rotation=45)
plt.yticks(rotation=45)
#plt.imshow(model_overlap_matrix[dataset], cmap = 'grey')
#plt.xticks(np.arange(len(models)), models, rotation = 45)
#plt.yticks(np.arange(len(models)), models, rotation = 45)
plt.show()
fig.savefig(join(rootsavedir,'RDM_compactness.png'))

In [ ]:
def sample_catrdm_pairs(cat_activations, submodels, n_samples=1000, nb_subcategories=12, nb_per_category = 50,
                                    batch_size=10, seed=None):
    """
    Memory-efficient version that processes in batches and optionally saves to disk.

    Parameters:
    -----------
    batch_size : int
        Number of samples to process at once (default: 1000)
    output_file : str, optional
        If provided, saves results to this file using pickle
    """

    if seed is not None:
        np.random.seed(seed)

    dissimilarity_metric = 'L2squared'

    nb_categories = len(cat_activations[submodels[0]])
    n_batches = (n_samples + batch_size - 1) // batch_size

    all_sims_samples = []
    all_indices = []
    print(f"Processing {n_samples} samples in {n_batches} batches of {batch_size}...")

    batch_rdms = {}
    for batch_idx in tqdm(range(n_batches)):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, n_samples)
        current_batch_size = end_idx - start_idx

        subset_size = nb_subcategories
        # Allocate batch arrays
        batch_sim = np.zeros((current_batch_size))
        batch_indices = np.zeros((current_batch_size, subset_size), dtype=int)
        for model in submodels:
            batch_rdms[model] = np.zeros((current_batch_size, nb_subcategories*nb_per_category, nb_subcategories*nb_per_category))
        for i in range(current_batch_size):
            # Randomly select images
            cat_indices = np.random.choice(nb_categories, size=nb_subcategories, replace=False)

            # Compute subrdms
            for model in submodels:
                batch_rdms[model][i] = rsa.compute_RDMs(cat_activations[model][cat_indices].reshape(nb_subcategories*nb_per_category, -1),
                            metric=dissimilarity_metric, display=False)
            # Extract submatrices
            batch_sim[i] = rsa.Compute_sim_RDMs(batch_rdms[submodels[0]][i], batch_rdms[submodels[1]][i], center = False, metric = 'pearson' )
            batch_indices[i] = cat_indices

        all_sims_samples.append(batch_sim)
        all_indices.append(batch_indices)

    # Concatenate all batches
    sim_samples = np.concatenate(all_sims_samples, axis=0)
    indices_used = np.concatenate(all_indices, axis=0)


    return sim_samples, indices_used




In [ ]:
'''nb_subcategories = 12
for i, model1 in enumerate(models[:-2]):
    for j, model2 in enumerate(models[i+1:-1]): # ignore random
        sim_samples, indices_used = sample_catrdm_pairs(cat_activations, [model1, model2], n_samples=100, nb_subcategories = nb_subcategories, nb_per_category = 50, batch_size=10, seed=None)
        np.save(f'results/categories_sim_samples_ohran_{model1}_{model2}_{dataset}.npy', sim_samples)'''

In [ ]:
def subsimilar_categories(cat_activations, submodels, dissimilarity_metric = 'L2squared', similarity_metric = 'pearson', nb_subcategories = 12):
    assert len(submodels)== 2
    assert cat_activations[submodels[0]].shape[:2] == cat_activations[submodels[1]].shape[:2]

    shape = cat_activations[submodels[0]].shape

    nb_categories = shape[0]
    nb_per_categories = shape[1]

    mean_cat_activations1 = cat_activations[submodels[0]].mean(axis = 1)
    mean_cat_activations2 = cat_activations[submodels[1]].mean(axis = 1)

    RDM1 = rsa.compute_RDMs(mean_cat_activations1,
                            metric=dissimilarity_metric, display=False)
    RDM2 = rsa.compute_RDMs(mean_cat_activations2,
                            metric=dissimilarity_metric, display=False)

    # exclude diagonal
    RDM1_short = np.array([np.delete(RDM1[i], i) for i in range(len(RDM1))]).transpose()
    RDM2_short = np.array([np.delete(RDM2[i], i) for i in range(len(RDM2))]).transpose()
    #center
    RDM1_centered = RDM1_short - np.mean(RDM1_short)
    RDM2_centered = RDM2_short - np.mean(RDM2_short)

    #RDM1_centered = RDM1_short
    #RDM2_centered = RDM2_short

    RDM1_centered = RDM1_centered / np.sqrt(np.sum(RDM1_centered ** 2, axis = 0))
    RDM2_centered = RDM2_centered / np.sqrt(np.sum(RDM2_centered ** 2, axis = 0))

    correlations = np.sum(RDM1_centered * RDM2_centered, axis=0)
    subsimiliar_categories = np.argsort(correlations)[:nb_subcategories]


    return correlations, subsimiliar_categories




In [ ]:
cat_similarities = {}
correlations = {}
similarities = {}
for i, model1 in enumerate(models[:-1]):
    for j, model2 in enumerate(models[i+1:]):
        print(f'{model1} vs {model2}')
        correlations[f'{model1}_{model2}'], subsimilar_cats = subsimilar_categories(cat_activations, [model1, model2], nb_subcategories = nb_subcategories)
        RDM1, RDM2, RDM1_sorted, RDM2_sorted, sorted_indices = max_rsa.find_subsimilar_subset(cat_activations, [model1, model2], subsimilar_cats,  images_per_subset = 4, nb_per_category = 50)
        cat_sim = rsa.Compute_sim_RDMs(RDM1, RDM2, metric = 'pearson')
        sim = rsa.Compute_sim_RDMs(RDM1_sorted, RDM2_sorted, metric = 'pearson')

        ''')fig, subs = plt.subplots(1,2, sharex=True, sharey=True)
        subs[0].imshow(RDM1, cmap='gray')
        subs[1].imshow(RDM2, cmap='gray')
        subs[0].axis('off')
        subs[1].axis('off')
        fig.suptitle(f'{cat_sim}')
        fig.tight_layout()'''
        cat_similarities[f'{model1}_{model2}'] = cat_sim
        print(np.array(listcat)[subsimilar_cats])

        savename = f'corr_{model1}_{model2}'
        fig, subs = plt.subplots(1,2, sharex=True, sharey=True)
        subs[0].imshow(RDM1_sorted, cmap='gray')
        subs[1].imshow(RDM2_sorted, cmap='gray')
        subs[0].axis('off')
        subs[1].axis('off')
        fig.suptitle(f'{sim}')
        fig.tight_layout()
        plt.show()
        fig.savefig(join(rootsavedir, f'subRDMs/{savename}.png'))
        plt.close()
        similarities[f'{model1}_{model2}'] = sim

In [ ]:
nb_seleted_categories = nb_subcategories
cat_similarities_compact = {}
similarities_compact = {}
sorted_indices = {}
maxdiffs = {}
labels_maxcompact = {}
for i, model1 in enumerate(models[:-1]):
    for j, model2 in enumerate(models[i+1:]):
        print(f'{model1} vs {model2}')
        labels_maxcompact[f'{model1}_{model2}'], sortedmaxdiffcats, maxdiffs[f'{model1}_{model2}'] = max_rsa.max_compactness_difference(
                sorted_compact_categories, compactness, listcat, models = [model1, model2],
                nb_considered_categories = nb_seleted_categories, compactness_diff_measure = 'normalizedDiff'
            )
        RDM1, RDM2, RDM1_sorted, RDM2_sorted, sorted_indices[f'{model1}_{model2}'] = max_rsa.find_subsimilar_subset(cat_activations, [model1, model2], labels_maxcompact[f'{model1}_{model2}'][:nb_seleted_categories],  images_per_subset = 4, nb_per_category = 50)
        cat_sim = rsa.Compute_sim_RDMs(RDM1, RDM2, metric = 'pearson')
        sim = rsa.Compute_sim_RDMs(RDM1_sorted, RDM2_sorted, metric = 'pearson')
        ''')fig, subs = plt.subplots(1,2, sharex=True, sharey=True)
        subs[0].imshow(RDM1, cmap='gray')
        subs[1].imshow(RDM2, cmap='gray')
        subs[0].axis('off')
        subs[1].axis('off')
        fig.suptitle(f'{rsa.Compute_sim_RDMs(RDM1, RDM2, metric = 'pearson')}')
        fig.tight_layout()'''
        cat_similarities_compact[f'{model1}_{model2}'] = cat_sim

        savename = f'compact_{model1}_{model2}'
        fig, subs = plt.subplots(1,2, sharex=True, sharey=True)
        subs[0].imshow(RDM1_sorted, cmap='gray')
        subs[1].imshow(RDM2_sorted, cmap='gray')
        subs[0].axis('off')
        subs[1].axis('off')
        fig.suptitle(f'{rsa.Compute_sim_RDMs(RDM1_sorted, RDM2_sorted, metric = 'pearson')}')
        fig.tight_layout()
        fig.savefig(join(rootsavedir,f'subRDMs/{savename}.png'))
        plt.show()
        plt.close()
        similarities_compact[f'{model1}_{model2}'] = sim


In [ ]:
import glob
listsamples = glob.glob(f'results/categories_sim_samples_ohran_*_{dataset}.npy')
nb_cols = 5
count = 0
fig, subs = plt.subplots(nrows = 2, ncols = nb_cols, figsize = (25,10), sharex = True, sharey = True)
for i, model1 in enumerate(models[:-2]):
    for j, model2 in enumerate(models[i+1:-1]):
        name = f'{model1}_{model2}'
        sample = np.load(f'results/categories_sim_samples_ohran_{name}_{dataset}.npy')
        hist, bin_edges = np.histogram(sample, 20)
        subs[count//5, count%5].bar(bin_edges[:-1],hist/max(hist), width = bin_edges[1] - bin_edges[0], linewidth = 0, align = 'edge')
        #subs[f//5, f%5].legend()
        subs[count//5, count%5].set_xlabel('Similarity')
        subs[count//5, count%5].set_ylabel('Density')
        subs[count//5, count%5].set_title(name)
        subs[count//5, count%5].vlines(cat_similarities[name],0,1, 'g')
        subs[count//5, count%5].vlines(cat_similarities_compact[name],0,1, 'r')
        count += 1


plt.tight_layout()
plt.show()
#fig.savefig(f'figures/compactness/categories_sim_samples_{arch}_{model1}_{model2}_{dataset}.npy')
plt.close()

In [ ]:
#### Compute correlations between the model's
nb_cols = 5
fig, subs = plt.subplots(2,nb_cols, sharex=True, sharey=True, figsize = (25, 10))
count = 0

for i, model1 in enumerate(models[:-2]):
    for j, model2 in enumerate(models[i+1:-1]):
        #print(f'{model1} vs {model2}')
        x = np.absolute(maxdiffs[f'{model1}_{model2}'])[np.argsort(labels_maxcompact[f'{model1}_{model2}'])]
        y = correlations[f'{model1}_{model2}']
        subs[count//nb_cols, count%nb_cols].scatter(x, y, color = 'k', s = 3)
        corr = np.round(np.corrcoef(x, y)[0,1], 2)
        subs[count//nb_cols, count%nb_cols].set_title(f'{model1} vs {model2}: {corr}')
        subs[count//nb_cols, count%nb_cols].set_xlabel('Compactness diff')
        subs[0,0].set_ylabel('Mean correlation')
        count+=1
fig.tight_layout()
fig.savefig(join(rootsavedir,'Compactness Vs Correlation.png'))
plt.show()
plt.close()

In [ ]:
### let's try to combine both for a look!
cats_correlations_compact = {}
indexes_corr_compact = {}
for i, model1 in enumerate(models[:-1]):
    for j, model2 in enumerate(models[i+1:]):
        x = np.absolute(maxdiffs[f'{model1}_{model2}'])[np.argsort(labels_maxcompact[f'{model1}_{model2}'])]
        y = correlations[f'{model1}_{model2}']
        score = x-y # maximizing compactness difference an minimizing correlation
        indexes_corr_compact[f'{model1}_{model2}'] = np.argsort(-score)
        cats_correlations_compact[f'{model1}_{model2}'] = np.array(listcat)[np.argsort(-score)]


In [ ]:
cat_similarities_corr_compact = {}
similarities_corr_compact = {}
sorted_indices_corr_compact = {}
controversial_RDMs_corr_compact = {}
images_per_subset = 4
for i, model1 in enumerate(models[:-1]):
    for j, model2 in enumerate(models[i+1:]):
        print(f'{model1} vs {model2}')
        RDM1, RDM2, RDM1_sorted, RDM2_sorted, sorted_indices_corr_compact[f'{model1}_{model2}'] = max_rsa.find_subsimilar_subset(cat_activations, [model1, model2], indexes_corr_compact[f'{model1}_{model2}'][:nb_subcategories],  images_per_subset = images_per_subset, nb_per_category = 50)
        controversial_RDMs_corr_compact[f'{model1}_{model2}'] = []
        controversial_RDMs_corr_compact[f'{model1}_{model2}'].append(RDM1_sorted)
        controversial_RDMs_corr_compact[f'{model1}_{model2}'].append(RDM2_sorted)
        cat_sim = rsa.Compute_sim_RDMs(RDM1, RDM2, metric = 'pearson')
        sim = rsa.Compute_sim_RDMs(RDM1_sorted, RDM2_sorted, metric = 'pearson')

        ''')fig, subs = plt.subplots(1,2, sharex=True, sharey=True)
        subs[0].imshow(RDM1, cmap='gray')
        subs[1].imshow(RDM2, cmap='gray')
        subs[0].axis('off')
        subs[1].axis('off')
        fig.suptitle(f'{cat_sim}')
        fig.tight_layout()'''
        cat_similarities_corr_compact[f'{model1}_{model2}'] = cat_sim
        print(cats_correlations_compact[f'{model1}_{model2}'][:nb_subcategories])

        savename = f'corr_compact_{model1}_{model2}'
        fig, subs = plt.subplots(1,2, sharex=True, sharey=True)
        subs[0].imshow(RDM1_sorted, cmap='gray')
        subs[1].imshow(RDM2_sorted, cmap='gray')
        subs[0].axis('off')
        subs[1].axis('off')
        fig.suptitle(f'{sim}')
        fig.tight_layout()
        plt.show()
        fig.savefig(join(rootsavedir,f'subRDMs/{savename}.png'))
        plt.close()
        similarities_corr_compact[f'{model1}_{model2}'] = sim

In [ ]:
import glob
listsamples = glob.glob(f'results/categories_sim_samples_ohran_*_{dataset}.npy')
nb_cols = 5
count = 0
fig, subs = plt.subplots(nrows = 2, ncols = nb_cols, figsize = (25,10), sharex = True, sharey = True)
for i, model1 in enumerate(models[:-2]):
    for j, model2 in enumerate(models[i+1:-1]):
        name = f'{model1}_{model2}'
        sample = np.load(f'results/categories_sim_samples_ohran_{name}_{dataset}.npy')
        hist, bin_edges = np.histogram(sample, 20)
        subs[count//5, count%5].bar(bin_edges[:-1],hist/max(hist), width = bin_edges[1] - bin_edges[0], linewidth = 0, align = 'edge')
        #subs[f//5, f%5].legend()
        subs[count//5, count%5].set_xlabel('Similarity')
        subs[count//5, count%5].set_ylabel('Density')
        subs[count//5, count%5].set_title(name)
        subs[count//5, count%5].vlines(cat_similarities[name],0,1, 'g')
        subs[count//5, count%5].vlines(cat_similarities_compact[name],0,1, 'r')
        subs[count//5, count%5].vlines(cat_similarities_corr_compact[name],0,1, 'y')
        count += 1


plt.tight_layout()
plt.show()
fig.savefig(join(rootsavedir,'categories_sim_samples.png'))
plt.close()

In [ ]:
## Looking at selections
imagelist = [img.replace('/raid/leonard_vandyck/datasets/genloc/', '/home/alban/Documents/datasets/genLOC2/') for img in imagelist]
#imagelist = [img.replace('/raid/shared/datasets/visoin/ecoset/', '/home/alban/Documents/ecoset/') for img in imagelist]
imagespaths = {}
for i, model1 in enumerate(models[:-1]):
    for j, model2 in enumerate(models[i+1:]):
        savename = f'Truenormalize_Fisher_discriminant_corr_compact_ohran_{model1}_{model2}_{dataset}'
        images, imagespaths[model1 + '_' + model2] = max_rsa.display_low_similarity_images(imagelist, sorted_indices_corr_compact[f'{model1}_{model2}'], maxdiffs[f'{model1}_{model2}'][:nb_seleted_categories], n_images=48,
                                                      grid_cols=8, figsize=(20, 12),
                                                      save_path=join(rootsavedir, f'subsets/{savename}.png'))

In [ ]:
RDMs = {}
metric = 'L2squared'
for i, model in enumerate(models):
    print(model)
    RDMs[model] = rsa.compute_RDMs(activations[model], metric = metric, display = False, title = f'{model}_{metric}')

In [ ]:
def sample_rdm_pairs(RDM1, RDM2, n_samples=100000, subset_size=40,
                                    batch_size=10000, seed=None):
    """
    Memory-efficient version that processes in batches and optionally saves to disk.

    Parameters:
    -----------
    batch_size : int
        Number of samples to process at once (default: 1000)
    output_file : str, optional
        If provided, saves results to this file using pickle
    """

    if seed is not None:
        np.random.seed(seed)

    n_images = RDM1.shape[0]
    n_batches = (n_samples + batch_size - 1) // batch_size

    all_sims_samples = []
    all_indices = []
    print(f"Processing {n_samples} samples in {n_batches} batches of {batch_size}...")

    for batch_idx in tqdm(range(n_batches)):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, n_samples)
        current_batch_size = end_idx - start_idx

        # Allocate batch arrays
        batch_sim = np.zeros((current_batch_size))
        batch_indices = np.zeros((current_batch_size, subset_size), dtype=int)

        for i in range(current_batch_size):
            # Randomly select images
            indices = np.random.choice(n_images, size=subset_size, replace=False)
            indices = np.sort(indices)

            # Extract submatrices
            batch_sim[i] = rsa.Compute_sim_RDMs(RDM1[np.ix_(indices, indices)], RDM2[np.ix_(indices, indices)], center = False, metric = 'pearson' )
            batch_indices[i] = indices

        all_sims_samples.append(batch_sim)
        all_indices.append(batch_indices)

    # Concatenate all batches
    sim_samples = np.concatenate(all_sims_samples, axis=0)
    indices_used = np.concatenate(all_indices, axis=0)


    return sim_samples, indices_used




In [ ]:
'''for i, model1 in enumerate(models[:-2]):
    for j, model2 in enumerate(models[i+1:-1]): # ignore random
        sim_samples, indices_used = sample_rdm_pairs(RDMs[model1], RDMs[model2], n_samples=100000, subset_size=48,
                                    batch_size=2000, seed=None)
        np.save(f'results/sim_samples_ohran_{model1}_{model2}_{dataset}.npy', sim_samples)'''

In [ ]:
import glob
listsamples = glob.glob(f'results/sim_samples_ohran_*_{dataset}.npy')
nb_cols = 5
count = 0
fig, subs = plt.subplots(nrows = 2, ncols = nb_cols, figsize = (25,10), sharex = True, sharey = True)
for i, model1 in enumerate(models[:-2]):
    for j, model2 in enumerate(models[i+1:-1]):
        name = f'{model1}_{model2}'
        sample = np.load(f'results/sim_samples_ohran_{name}_{dataset}.npy')
        hist, bin_edges = np.histogram(sample, 50)
        subs[count//5, count%5].bar(bin_edges[:-1],hist/max(hist), width = bin_edges[1] - bin_edges[0], linewidth = 0, align = 'edge')
        #subs[f//5, f%5].legend()
        subs[count//5, count%5].set_xlabel('Similarity')
        subs[count//5, count%5].set_ylabel('Density')
        subs[count//5, count%5].set_title(name)
        subs[count//5, count%5].vlines(similarities[name],0,1, 'g')
        subs[count//5, count%5].vlines(similarities_compact[name],0,1, 'r')
        subs[count//5, count%5].vlines(similarities_corr_compact[name],0,1, 'orange')
        count += 1


plt.tight_layout()
plt.show()
fig.savefig(join(rootsavedir,'sim_samples.png'))
plt.close()

In [ ]:
### Work towards t-sne visualization

# Main execution
import lib.visualization_sim as vis
importlib.reload(vis)

# For demonstration, run with synthetic data
for i, model1 in enumerate(models[:-1]):
    for j, model2 in enumerate(models[i+1:]):
        name = f'{model1}_{model2}'
        labels = np.array(listcat)[indexes_corr_compact[f'{model1}_{model2}'][:nb_subcategories]]
        labels = np.repeat(labels, images_per_subset)
        tsne_results, labels, fig = vis.model_comparison_tsne_pipeline(controversial_RDMs_corr_compact[name][0],controversial_RDMs_corr_compact[name][1], labels, name)
        fig.savefig(join(rootsavedir, f'tSNE/{name}.png'))

plt.close()





